# Notebook 14: Evaluation of Fine-tuned Models

## Overview

- **Duration**: ~1.5 hours
- **Prerequisites**: Notebook 13 (Fine-tuning)
- **Learning Objectives**:
  1. Compute perplexity to measure model quality
  2. Compare base model vs fine-tuned model
  3. Perform qualitative evaluation with test prompts
  4. Understand evaluation best practices

## Introduction

### Why Evaluation Matters

After fine-tuning, we need to verify:
- Model has learned the new task/domain
- Model hasn't degraded on general capabilities
- Model follows the desired format and behavior

### Evaluation Methods

| Method | Measures | Automated? |
|--------|----------|------------|
| Perplexity | How well model predicts text | Yes |
| Loss | Training/validation loss | Yes |
| Benchmarks | Performance on standard tasks | Yes |
| Human eval | Quality, helpfulness, safety | No |

In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/yani/DL_internship_week2/ve_dlworkshopuv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Step 1: Load Models for Comparison

We'll compare the base model with our fine-tuned version.

In [2]:
##check what is using GPU if there is any
#nvidia-smi
##check if the GPU is available
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 1
GPU: NVIDIA GeForce RTX 4070


In [3]:
from unsloth import FastLanguageModel

# Load base model
#base_model_name = "unsloth/Llama-3.2-3B-bnb-4bit"
base_model_name = "/data/Ministral-3-14B-Instruct-2512"
max_seq_length = 2048

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    device_map="cuda",   # add this just for small GPU machine
)

print("Base model loaded!")

/tmp/ipykernel_192277/122118340.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.719 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Base model loaded!


In [4]:
# Load fine-tuned model (with LoRA adapter)
finetuned_model, _ = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Load LoRA weights
from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(finetuned_model, "my_lora_adapter")

print("Fine-tuned model loaded!")

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.719 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Fine-tuned model loaded!


## Exercise 14.1: Compute Perplexity (30 min)

Perplexity measures how well a model predicts text. Lower is better.

$$\text{Perplexity} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(x_i | x_{<i})\right)$$

In practice: `perplexity = exp(average_loss)`

In [ ]:
####solution 14.1
@torch.no_grad()
def compute_perplexity(
    model,
    tokenizer,
    texts: list[str],
    max_length: int = 512,
    batch_size: int = 4,
) -> float:
    """
    Compute perplexity of model on a list of texts.
    
    Args:
        model: The language model
        tokenizer: Tokenizer for the model
        texts: List of text strings to evaluate
        max_length: Maximum sequence length
        batch_size: Batch size for evaluation
        
    Returns:
        Perplexity value (float)
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    # TODO: Implement perplexity computation
    # 1. Iterate through texts in batches
    for i in range(0, len(texts), batch_size):
        # 2. Tokenize each batch
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(model.device)
        # 3. Compute loss (cross-entropy) for each batch
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()  # Get scalar loss value
         # 4. Accumulate total loss and token count
        total_loss += loss * inputs["input_ids"].numel()  # Multiply by number of tokens in batch
        total_tokens += inputs["input_ids"].numel()
   
    # 5. Return exp(total_loss / total_tokens)
    #
    # Hint: For loss computation:
    #   outputs = model(input_ids, labels=input_ids)
    #   loss = outputs.loss  # This is average loss per token
    
    return np.exp(total_loss / total_tokens)

In [6]:
####test cell
# Create evaluation dataset
#eval_dataset = load_dataset("yahma/alpaca-cleaned", split="train[:500]")
eval_dataset = load_dataset("/data/alpaca-cleaned", split="train[:500]")

# Format for evaluation
def format_for_eval(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    user_message = instruction
    if input_text and input_text.strip():
        user_message = f"{instruction}\n\n{input_text}"
    
    text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>"
    text += f"<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
    return text

eval_texts = [format_for_eval(ex) for ex in eval_dataset]
print(f"Evaluation texts: {len(eval_texts)}")

Evaluation texts: 500


In [7]:
####test cell
# Compute perplexity for both models
FastLanguageModel.for_inference(base_model)
base_ppl = compute_perplexity(base_model, tokenizer, eval_texts[:100])
print(f"Base model perplexity: {base_ppl:.2f}")

FastLanguageModel.for_inference(finetuned_model)
finetuned_ppl = compute_perplexity(finetuned_model, tokenizer, eval_texts[:100])
print(f"Fine-tuned model perplexity: {finetuned_ppl:.2f}")

print(f"\nImprovement: {(base_ppl - finetuned_ppl) / base_ppl * 100:.1f}%")

Computing perplexity: 100%|███████████████████████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  4.04it/s]


Base model perplexity: 133.90


Computing perplexity: 100%|███████████████████████████████████████████████████████████████████████████████| 25/25 [00:06<00:00,  4.03it/s]

Fine-tuned model perplexity: 31.63

Improvement: 76.4%


## Exercise 14.2: Side-by-Side Comparison (30 min)

Compare generations from both models on the same prompts.

In [ ]:
####solution 14.2
@torch.no_grad()
def generate_response(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
) -> str:
    """
    Generate a response from the model.
    
    Args:
        model: Language model
        tokenizer: Tokenizer
        prompt: User prompt
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        
    Returns:
        Generated response string
    """
    # TODO: Implement generation
    # 1. Format prompt with Llama 3 template
        # Example template:
    # formatted_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    formatted_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    # 2. Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    # 3. Generate with model.generate()
    outputs = model.generate(**inputs, 
                             max_new_tokens=max_new_tokens,
                             temperature=temperature,  # Optional: adjust for more creative (higher) or focused (lower) responses
                             do_sample=True,  # Enable sampling for more varied responses
    )
    # 4. Decode and return response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [9]:
####test cell
def compare_models(prompt: str):
    """Compare responses from base and fine-tuned models."""
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    
    # Base model response
    FastLanguageModel.for_inference(base_model)
    base_response = generate_response(base_model, tokenizer, prompt)
    print(f"\n--- BASE MODEL ---")
    print(base_response)
    
    # Fine-tuned model response
    FastLanguageModel.for_inference(finetuned_model)
    ft_response = generate_response(finetuned_model, tokenizer, prompt)
    print(f"\n--- FINE-TUNED MODEL ---")
    print(ft_response)

In [ ]:
####test cell
test_prompts = [
    "Explain how a neural network learns.",
    "Write a haiku about Python programming.",
    "What are the benefits of version control?",
    "Translate 'Hello, how are you?' to French.",
    "Summarize the key points of machine learning.",
]

for prompt in test_prompts:
    compare_models(prompt)

## Exercise 14.3: Response Quality Metrics (20 min)

Compute basic quality metrics for generated responses.

In [ ]:
####solution 14.3
def compute_response_metrics(response: str) -> dict:
    """
    Compute basic quality metrics for a response.
    
    Args:
        response: Generated text
        
    Returns:
        Dictionary of metrics
    """
    # TODO: Implement basic metrics
    dict_metrics = {}
    # - Length (words and characters)
    dict_metrics["length_words"] = len(response.split())
    dict_metrics["length_characters"] = len(response)
    # - Sentence count
    dict_metrics["sentence_count"] = response.count(".")
    # - Average word length
    words = response.split()
    if words:
        dict_metrics["avg_word_length"] = sum(len(word) for word in words) / len(words)
    else:
        dict_metrics["avg_word_length"] = 0
    # - Unique words ratio (vocabulary diversity)
    unique_words = set(words)
    if words:
        dict_metrics["unique_words_ratio"] = len(unique_words) / len(words)
    else:
        dict_metrics["unique_words_ratio"] = 0

    return dict_metrics

In [12]:
####test cell
# Compare metrics across multiple prompts
def evaluate_model_quality(model, tokenizer, prompts, model_name="Model"):
    """Evaluate a model on multiple prompts and aggregate metrics."""
    all_metrics = []
    
    FastLanguageModel.for_inference(model)
    
    for prompt in tqdm(prompts, desc=f"Evaluating {model_name}"):
        response = generate_response(model, tokenizer, prompt)
        metrics = compute_response_metrics(response)
        all_metrics.append(metrics)
    
    # Aggregate metrics
    aggregated = {}
    for key in all_metrics[0].keys():
        values = [m[key] for m in all_metrics]
        aggregated[key] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'min': np.min(values),
            'max': np.max(values),
        }
    
    return aggregated

In [ ]:
####test cell
# Run evaluation (using a subset of prompts)
eval_prompts = [
    "Explain recursion in programming.",
    "What is the difference between a list and a tuple in Python?",
    "Write a function to check if a string is a palindrome.",
    "Describe the MVC design pattern.",
    "What are the SOLID principles?",
]

base_metrics = evaluate_model_quality(base_model, tokenizer, eval_prompts, "Base")
ft_metrics = evaluate_model_quality(finetuned_model, tokenizer, eval_prompts, "Fine-tuned")

print("\nBase Model Metrics:")
for metric, stats in base_metrics.items():
    print(f"  {metric}: {stats['mean']:.2f} (+/- {stats['std']:.2f})")

print("\nFine-tuned Model Metrics:")
for metric, stats in ft_metrics.items():
    print(f"  {metric}: {stats['mean']:.2f} (+/- {stats['std']:.2f})")

## Exercise 14.4: Instruction Following Test (20 min)

Test if the model properly follows specific instructions.

In [14]:
# Instruction following tests
instruction_tests = [
    {
        "prompt": "List exactly 3 programming languages.",
        "check": lambda r: len([line for line in r.split('\n') if line.strip()]) >= 3,
        "description": "Should list at least 3 items"
    },
    {
        "prompt": "Respond with only one word: What color is the sky?",
        "check": lambda r: len(r.split()) <= 3,  # Allow some slack
        "description": "Should give a short response"
    },
    {
        "prompt": "Write a sentence that starts with 'The'.",
        "check": lambda r: r.strip().lower().startswith('the'),
        "description": "Should start with 'The'"
    },
    {
        "prompt": "What is 2+2? Answer with just the number.",
        "check": lambda r: '4' in r,
        "description": "Should contain '4'"
    },
]

In [15]:
def run_instruction_tests(model, tokenizer, tests, model_name="Model"):
    """Run instruction following tests on a model."""
    FastLanguageModel.for_inference(model)
    
    results = []
    for test in tests:
        response = generate_response(model, tokenizer, test["prompt"], max_new_tokens=100)
        passed = test["check"](response)
        results.append({
            "prompt": test["prompt"],
            "description": test["description"],
            "response": response,
            "passed": passed,
        })
    
    # Print results
    print(f"\n{'='*60}")
    print(f"INSTRUCTION FOLLOWING TEST: {model_name}")
    print(f"{'='*60}")
    
    passed_count = sum(r["passed"] for r in results)
    print(f"\nScore: {passed_count}/{len(results)} ({passed_count/len(results)*100:.0f}%)\n")
    
    for r in results:
        status = "✓" if r["passed"] else "✗"
        print(f"{status} {r['description']}")
        print(f"  Prompt: {r['prompt']}")
        print(f"  Response: {r['response'][:100]}..." if len(r['response']) > 100 else f"  Response: {r['response']}")
        print()
    
    return results

In [ ]:
####solution 14.4
# TODO: Run instruction tests on both models: base_model and finetuned_model
base_test_results = run_instruction_tests(base_model, tokenizer, instruction_tests, "Base")
ft_test_results = run_instruction_tests(finetuned_model, tokenizer, instruction_tests, "Fine-tuned")

## Summary

### What You Learned

1. **Perplexity**: Quantitative measure of model quality
2. **Side-by-side comparison**: Visual quality assessment
3. **Response metrics**: Length, diversity, format checking
4. **Instruction following**: Testing specific behaviors

### Evaluation Best Practices

- Use multiple evaluation methods
- Test on held-out data (not training data)
- Include edge cases and adversarial prompts
- Compare against baselines
- Consider human evaluation for quality

### Next: Export

In Notebook 15, we'll export our model for deployment!